In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
import os
from time import sleep
from pathlib import Path
from urllib.parse import quote, unquote, urlparse, parse_qs
import numpy as np
from openpyxl import load_workbook

from birddog.wiki import (
    get_title,
    WIKI_NAMESPACE,
    download_thumbnail,
    )

from birddog.nocodb import (
    process_archive_sheet,
    process_fond_sheet,
    process_opus_sheet,
    process_worksheets,
    NocoDBClient,
    )

2026-01-01 09:30:54,002 [INFO] Translation is enabled. Using GCP translator
2026-01-01 09:30:54,003 [INFO] Using Google Cloud translation API
2026-01-01 09:30:54,004 [INFO] GoogleCloudTranslator using REST API


In [3]:
root = "./var/WAT"

In [4]:
def list_files(directory: str, suffix: str, prefix=None) -> list[Path]:
    """Recursively list only files in `directory` with the given `suffix`,
    skipping any whose name starts with '~'.
    """
    return [
        p for p in Path(directory).rglob(f"*{suffix}")
        if p.is_file() and not p.name.startswith("~")
        if not prefix or p.name.startswith(prefix)
    ]

In [5]:
list_files(root, "xlsx")

[PosixPath('var/WAT/DARO (Rivne)/DARO-D-wiki-20250915.xlsx'),
 PosixPath('var/WAT/DARO (Rivne)/DARO-R-wiki-20250728.xlsx'),
 PosixPath('var/WAT/DAVO (Volhynia)/DAVO-archives+AK-20250525b.xlsx'),
 PosixPath('var/WAT/DAVO (Volhynia)/DAVO-D-wiki-20250915.xlsx'),
 PosixPath('var/WAT/DAVO (Volhynia)/DAVO-R-wiki-20250915.xlsx'),
 PosixPath('var/WAT/DAHO (Kharkiv)/DAHO-archive-20250719.xlsx'),
 PosixPath('var/WAT/DAHO (Kharkiv)/DAHO-R-wiki-20250908.xlsx'),
 PosixPath('var/WAT/DAHO (Kharkiv)/DAHO-D-wiki-20250817a.xlsx'),
 PosixPath('var/WAT/DACHKO (Cherkassy)/DACHKO-R-wiki-20250904.xlsx'),
 PosixPath('var/WAT/DACHKO (Cherkassy)/DACHKO-D-wiki-20250904.xlsx'),
 PosixPath('var/WAT/DAHEO (Kherson)/DAHEO-R-archive-20250819.xlsx'),
 PosixPath('var/WAT/DAHEO (Kherson)/DAHEO-P-wiki-20250708.xlsx'),
 PosixPath('var/WAT/DAHEO (Kherson)/DAHEO-D-wiki-20250821.xlsx'),
 PosixPath('var/WAT/DAHEO (Kherson)/DAHEO-R-wiki-20250724.xlsx'),
 PosixPath('var/WAT/DAHEO (Kherson)/DAHEO-D-archive-20250707.xlsx'),
 Posi

In [6]:
#wb = load_workbook(list_files(root, "xlsx", prefix="DAHEO-D-wiki")[0])

In [7]:
client = NocoDBClient()

2026-01-01 09:31:13,233 [INFO] pages loaded (819 records)
2026-01-01 09:31:40,113 [INFO] documents loaded (465 records)


In [8]:
#client.upsert_pages(process_worksheets(wb.worksheets[1:4]))

In [9]:
#client.upsert_pages(process_worksheets(wb.worksheets[4:6]))

In [10]:
#client.upsert_pages(process_worksheets(wb.worksheets[:1]))

In [11]:
#client.upsert_pages(process_worksheets(wb.worksheets[6:]))

In [12]:
#wb = load_workbook(list_files(root, "xlsx", prefix="DAHEO-P-wiki")[0])

In [13]:
#wb.worksheets

In [14]:
#client.upsert_pages(process_worksheets(wb.worksheets, page_table={}))

In [15]:
#wb = load_workbook(list_files(root, "xlsx", prefix="DAHEO-R-wiki")[0])

In [16]:
#client.upsert_pages(process_worksheets(wb.worksheets, page_table={}))

In [ ]:
docs = client.list_records("documents")

In [ ]:
doc_links = [item["link"] for item in docs if not item["doc_image"]]

In [ ]:
len(doc_links)

In [ ]:
#download_thumbnail(doc_links[1], out_dir="./var/thumbs")

In [ ]:
result = client.upload_attachment("./var/thumbs/ДАХеО_Фонд_1_Описи_1,_2.pdf.p1.w640.jpg")

In [ ]:
docs[1]

In [ ]:
download_result = download_thumbnail(docs[1]["link"], out_dir="./var/thumbs")
client.set_attachment("documents", docs[1]["link"], "doc_image", download_result["saved_path"])

In [ ]:
for doc in docs:
    if not doc["doc_image"]:
        print(f"Trying {doc["title"]}...")
        try:
            download_result = download_thumbnail(doc["link"], out_dir="./var/thumbs")
            client.set_attachment("documents", doc["link"], "doc_image", download_result["saved_path"])
        except Exception as e:
            print(f"failed: {e}")

In [ ]:
list_files(root, "xlsx", prefix="DAHO")

In [17]:
archives = ["DAHO-D", "DAHO-R"]

In [18]:
for archive in archives:
    file = list_files(root, "xlsx", prefix=f"{archive}-wiki")
    if file:
        print(file[0])

var/WAT/DAHO (Kharkiv)/DAHO-D-wiki-20250817a.xlsx
var/WAT/DAHO (Kharkiv)/DAHO-R-wiki-20250908.xlsx


In [19]:
def load_workbooks(prefix, only_wiki=True):
    files = list_files(root, "xlsx", prefix=prefix)
    return [load_workbook(str(file)) 
            for file in list_files(root, "xlsx", prefix=prefix)
            if "wiki" in str(file) or not only_wiki]

In [23]:
def upsert_archive(archive):
    wbs = load_workbooks(archive)
    if wbs:
        return process_worksheets(wbs[0].worksheets, page_table={})
        #client.upsert_pages()

In [24]:
recs=upsert_archive(archives[0])

2026-01-01 09:32:57,379 [INFO] processing worksheet: DAHO D wiki fund list
2026-01-01 09:32:57,392 [INFO] processing worksheet: fund 3
2026-01-01 09:32:57,392 [INFO] processing worksheet: DAHO 3-287
2026-01-01 09:32:57,393 [INFO] processing worksheet: fund 4
2026-01-01 09:32:57,394 [INFO] processing worksheet: DAHO 4-160
2026-01-01 09:32:57,394 [INFO] processing worksheet: DAHO 4-163
2026-01-01 09:32:57,395 [INFO] processing worksheet: DAHO 4-166
2026-01-01 09:32:57,396 [INFO] processing worksheet: DAHO 4-177
2026-01-01 09:32:57,397 [INFO] processing worksheet: DAHO 4-187
2026-01-01 09:32:57,398 [INFO] processing worksheet: fund 31
2026-01-01 09:32:57,398 [INFO] processing worksheet: DAHO 31-141
2026-01-01 09:32:57,406 [INFO] processing worksheet: fund 52
2026-01-01 09:32:57,407 [INFO] processing worksheet: DAHO 52-1
2026-01-01 09:32:57,408 [INFO] processing worksheet: DAHO 52-2
2026-01-01 09:32:57,408 [INFO] processing worksheet: fund 179
2026-01-01 09:32:57,409 [INFO] processing work

In [25]:
len(recs)


1097

In [26]:
type(recs)

dict

In [29]:
for k,v in recs.items():
    if not v.get("title"):
        print(k,v)

None {'title': None, 'label': 'General page content', 'level': 'case', 'description': None, 'years': None, 'availability': 'linked', 'source_type': 'wiki', 'parent': 'ДАХО/31/141', 'comments': None, 'doc_links': '', 'doc_type': None, 'content_code': None, 'process_code': None, 'processor': None, 'pages_processed': 0}


In [ ]:
w = load_workbooks("DAHO")

In [ ]:
w

In [ ]:
pages=process_worksheets(w[1].worksheets, page_table={})

In [ ]:
print([p["label"] for p in pages.values()])